# Physics baseline evaluation

Scores untouched base models on three exams with identical prompts and decoding, so you can pick the best base and later prove a fine-tune beat it.

**Before you start**
1. Put `physeval.py` and this notebook in one Google Drive folder, e.g. `MyDrive/physics_eval`.
2. In Colab choose **Runtime > Change runtime type > T4 GPU**.
3. Run the cells from top to bottom. Every step saves to Drive and continues where it stopped, so after a disconnect just run the cells again.

**Three model configurations** (set `RUN` in the first cell, one per Colab session):
- `lfm25-1.2b-thinking`: LiquidAI/LFM2.5-1.2B-Thinking
- `qwen35-2b-nothink`: Qwen/Qwen3.5-2B in its default mode
- `qwen35-2b-think`: Qwen/Qwen3.5-2B with thinking switched on

Qwen is tested in both modes because the LFM model always thinks; the better Qwen mode is the fair opponent.

## 1. Settings

In [ ]:
WORKDIR = "/content/drive/MyDrive/physics_eval"   # the Drive folder that holds physeval.py

RUN = "lfm25-1.2b-thinking"        # or "qwen35-2b-nothink" or "qwen35-2b-think"

# First pass: a balanced sample (about 1,260 UGPhysics + 356 PhysicsEval + 210 MATH-500 problems).
# For the full benchmarks set PER_GROUP = "" and run the generate cell again; finished items are reused.
PER_GROUP = "ugphysics=100,physicseval=25,math500=30"
BENCHMARKS = "ugphysics,physicseval,math500"

MAX_NEW_TOKENS = 4096        # official UGPhysics budget; keep identical for every run
REPETITION_PENALTY = 1.0     # official setting. For a second test use 1.05 with a new RUN name such as "qwen35-2b-think-rp105"
BATCH_SIZE = 16              # lowered automatically if the GPU runs out of memory
DTYPE = "auto"               # use "float32" only if the smoke test prints garbage

## 2. Mount Drive and check the GPU

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
assert os.path.exists(f"{WORKDIR}/physeval.py"), f"physeval.py not found in {WORKDIR}"
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 3. Install packages
If a Qwen3.5 load error appears later, run the commented line too and repeat the smoke test.

In [ ]:
!pip -q install -U transformers accelerate huggingface_hub datasets peft
!pip -q install "antlr4-python3-runtime==4.11.1" chardet
# !pip -q install "git+https://github.com/huggingface/transformers.git@main"

## 4. Check the grader
Must end with `SELF-TEST PASSED`. Do not trust any score until it does.

In [ ]:
!python "{WORKDIR}/physeval.py" selftest --workdir "{WORKDIR}"

## 5. Download the exams and freeze the questions
Runs once. Every model then gets exactly the same questions and prompts.

In [ ]:
!python "{WORKDIR}/physeval.py" prepare --workdir "{WORKDIR}" --benchmarks {BENCHMARKS}

## 6. Smoke test (about 5 to 10 minutes)
Runs 8 questions. Check four things in the output:
1. The printed prompt looks like a normal chat prompt.
2. The first completion is sensible physics, not garbage.
3. `padding ... "safe": true`. If it says false the harness switches to a slower, safe batching mode on its own.
4. The GPU-hours estimate, so you know how long the real run takes.

In [ ]:
!python "{WORKDIR}/physeval.py" generate --workdir "{WORKDIR}" --run {RUN} --smoke --max-items 8 \
    --benchmarks ugphysics --per-group "{PER_GROUP}" --max-new-tokens {MAX_NEW_TOKENS} \
    --batch-size {BATCH_SIZE} --dtype {DTYPE} --repetition-penalty {REPETITION_PENALTY}

## 7. Generate answers (the long step)
Safe to interrupt. After a disconnect run cells 1, 2, 3 and this one again; it continues from the last saved batch.

In [ ]:
!python "{WORKDIR}/physeval.py" generate --workdir "{WORKDIR}" --run {RUN} \
    --benchmarks {BENCHMARKS} --per-group "{PER_GROUP}" --max-new-tokens {MAX_NEW_TOKENS} \
    --batch-size {BATCH_SIZE} --dtype {DTYPE} --repetition-penalty {REPETITION_PENALTY}

## 8. Grade with the official rule-based judge
Free, no API key, needs no GPU. Grades every run found in the folder.

In [ ]:
!python "{WORKDIR}/physeval.py" grade --workdir "{WORKDIR}"

## 9. Optional: LLM judge
Needed for **PhysicsEval** (free-text answers cannot be graded by rules) and for the official two-stage **UGPhysics MARJ** score.

Works with any OpenAI-compatible endpoint. The example uses a Google AI Studio key: add it in Colab under the key icon (Secrets) as `JUDGE_API_KEY`. Use the **same judge model for every run**. Free tiers have daily limits; when the limit hits, the step stops and continues next time you run it.

In [ ]:
import os
from google.colab import userdata
os.environ["JUDGE_BASE_URL"] = "https://generativelanguage.googleapis.com/v1beta/openai/"
os.environ["JUDGE_MODEL"] = "gemini-3.8-flash"        # any chat model your key can use
os.environ["JUDGE_API_KEY"] = userdata.get("JUDGE_API_KEY")

!python "{WORKDIR}/physeval.py" judge --workdir "{WORKDIR}" --dry-run     # shows how many calls are needed

In [ ]:
!python "{WORKDIR}/physeval.py" judge --workdir "{WORKDIR}" --rpm 10

## 10. Report
Compares every graded run. Files are saved in `results/`: `summary.md`, `summary.json`, per-question CSVs and breakdown tables.

In [ ]:
!python "{WORKDIR}/physeval.py" report --workdir "{WORKDIR}" > /dev/null
from IPython.display import Markdown, display
display(Markdown(open(f"{WORKDIR}/results/summary.md").read()))

## Later: score your fine-tuned model the same way
Give it a new run name and point to your LoRA adapter (or a merged model with `--model`). Keep every other setting unchanged, then run cells 8 to 10 again.

In [ ]:
# !python "{WORKDIR}/physeval.py" generate --workdir "{WORKDIR}" --run my-finetune-v1 \
#     --model LiquidAI/LFM2.5-1.2B-Thinking --adapter your-hf-name/your-lora-adapter \
#     --benchmarks {BENCHMARKS} --per-group "{PER_GROUP}" --max-new-tokens {MAX_NEW_TOKENS} --batch-size {BATCH_SIZE} \
#     --repetition-penalty {REPETITION_PENALTY}